# Week 4 Challenge

Using different open source models to:
- Generate unit tests

**Models used** (via OpenRouter): Qwen2.5-Coder 32B, Qwen3 Coder, DeepSeek R1, GPT-OSS-20B

In [ ]:
import os
from dotenv import load_dotenv
import gradio as gr
from openai import OpenAI

In [ ]:
load_dotenv(override=True)

api_key = os.getenv("OPENROUTER_API_KEY")
if not api_key:
    raise ValueError("OPENROUTER_API_KEY not found")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)


In [ ]:

MODELS = [
    "qwen/qwen-2.5-coder-32b-instruct",
    "qwen/qwen3-coder",
    "deepseek/deepseek-r1",
    "openai/gpt-oss-20b"
]

In [ ]:
def build_prompt(chunk, language):
    frameworks = {"Python": "pytest", "JavaScript": "Jest", "Kotlin": "JUnit"}

    framework = frameworks[language]

    return f"""
        You are an expert Senior QA Engineer.

        Generate comprehensive unit tests for the following {language} code.
        Return ONLY the test code, no markdown formatting or explanations.

        Requirements:

        - Use the {framework} testing framework
        - Cover all classes and functions
        - Include edge cases
        - Include helpful comments
        - Output ONLY the test code

        - Generate ONLY the test code, no explanations
        - Include necessary imports
        - Test normal cases, edge cases, and error conditions
        - Use descriptive test names
        - Include assertions for expected behavior
        - Follow {language} best practices
        - Make tests complete and runnable

        CODE:
        {chunk}
        """

In [ ]:
def chunk_code(code, max_lines=120):
    lines = code.split("\n")
    chunks = []

    for i in range(0, len(lines), max_lines):
        chunks.append("\n".join(lines[i:i + max_lines]))

    return chunks

In [ ]:
def generate_tests(code, language, model):

    if not code.strip():
        return "Please paste code first."

    chunks = chunk_code(code)

    final_output = ""

    for i, chunk in enumerate(chunks):

        prompt = build_prompt(chunk, language)

        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You generate unit tests."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.3,
            max_tokens=1500
        )

        result = response.choices[0].message.content

        final_output += f"\n\n# ===== Tests for Section {i+1} =====\n"
        final_output += result

    return final_output

In [ ]:
def update_language(lang):

    mapping = {
        "Python": "python",
        "JavaScript": "javascript",
        "Kotlin": "kotlin"
    }

    return gr.Code(language=mapping[lang])

# UI

In [ ]:
with gr.Blocks() as demo:

    gr.Markdown("# 🧪 AI Unit Test Generator")

    with gr.Row():

        language = gr.Dropdown(
            ["Python", "JavaScript", "Kotlin"],
            value="Python",
            label="Programming Language"
        )

        model = gr.Dropdown(
            MODELS,
            value=MODELS[0],
            label="Model"
        )

    code_input = gr.Textbox(
        label="Paste Your Code (Supports 1000+ lines)",
        lines=25,
        placeholder="Paste functions or classes here..."
    )

    generate_button = gr.Button("Generate Unit Tests")

    output_tests = gr.Code(
        label="Generated Unit Tests",
        language="python",
        lines=30
    )

    generate_button.click(
        generate_tests,
        inputs=[code_input, language, model],
        outputs=[output_tests]
    )

demo.launch()